## 1. 교통량 데이터 탐색적 데이터 분석 (EDA)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 깨짐 방지
plt.rcParams['font.family'] = 'Malgun Gothic' if os.name == 'nt' else 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

### 1-1. 시계열 트래픽 분석 (시간 흐름에 따른 차량 대수 변화)

In [ ]:
df = pd.read_csv('c:\\AI_1team\\traffic_features.csv')

plt.figure(figsize=(12, 5))
plt.plot(df.index, df['car_count'], label='Car Count', color='blue', alpha=0.7)
plt.plot(df.index, df['bus_count'], label='Bus Count', color='green', alpha=0.7)
plt.plot(df.index, df['truck_count'], label='Truck Count', color='orange', alpha=0.7)

plt.title('Traffic Volume Over Time (Frames)')
plt.xlabel('Frame (Time)')
plt.ylabel('Vehicle Count')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### 1-2. 교통 밀집도(Density) 분포 확인

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['density'], bins=30, kde=True, color='purple')
plt.title('Distribution of Traffic Density')
plt.xlabel('Density (BBox Area / Image Area)')
plt.ylabel('Frequency')
plt.show()

## 1. YOLOv8 파라미터 및 모델 성능 벤치마크 (Nano vs Small vs Medium)

In [ ]:
import os
import glob
import time
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

### 1-1. 모델별 FPS(속도) 및 탐지율 비교 시각화

In [ ]:
search_path = r"c:\AI_1team\dataset\**\BC1001001\*.jpg"
image_paths = glob.glob(search_path, recursive=True)[:10] # 테스트용 10장

# 3가지 모델 로드
model_n = YOLO('yolov8n.pt')
model_s = YOLO('yolov8s.pt')
model_m = YOLO('yolov8m.pt')

results_dict = {'Nano': {'time': [], 'count': []}, 
                'Small': {'time': [], 'count': []}, 
                'Medium': {'time': [], 'count': []}}

for img_path in image_paths:
    img = cv2.imread(img_path)
    if img is None: continue
    
    for m_name, model in zip(['Nano', 'Small', 'Medium'], [model_n, model_s, model_m]):
        start = time.time()
        res = model(img, verbose=False)
        results_dict[m_name]['time'].append(time.time() - start)
        results_dict[m_name]['count'].append(sum([1 for box in res[0].boxes if int(box.cls[0]) in [2,5,7]]))

# 데이터프레임으로 결과 정리
summary_df = pd.DataFrame({
    'Model': ['Nano', 'Small', 'Medium'],
    'Avg Inference Time (s)': [np.mean(results_dict['Nano']['time']), np.mean(results_dict['Small']['time']), np.mean(results_dict['Medium']['time'])],
    'Avg FPS': [1/np.mean(results_dict['Nano']['time']), 1/np.mean(results_dict['Small']['time']), 1/np.mean(results_dict['Medium']['time'])],
    'Avg Detection Count': [np.mean(results_dict['Nano']['count']), np.mean(results_dict['Small']['count']), np.mean(results_dict['Medium']['count'])]
})
display(summary_df)

# 시각화
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.bar(summary_df['Model'], summary_df['Avg FPS'], color='skyblue', width=0.4, label='FPS (Speed)')
ax2.plot(summary_df['Model'], summary_df['Avg Detection Count'], color='red', marker='o', linewidth=2, label='Detection Count')

ax1.set_ylabel('FPS (Higher is faster)', color='blue')
ax2.set_ylabel('Detected Vehicles', color='red')
plt.title('YOLO Models Benchmark: Speed vs Detection Count')
plt.show()

### 1-2. Confidence Threshold에 따른 탐지 결과 시각화 비교 (0.25 vs 0.5 vs 0.75)

In [ ]:
if image_paths:
    img = cv2.imread(image_paths[0])
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, conf in zip(axes, [0.25, 0.5, 0.75]):
        res = model_n(img, conf=conf, verbose=False)
        plotted = res[0].plot()
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Confidence: {conf}')
        ax.axis('off')
    plt.show()

## 1. 이상 탐지(Anomaly Detection) 알고리즘 벤치마크 및 시각화
(Autoencoder vs Isolation Forest vs One-Class SVM)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, recall_score
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
import warnings
warnings.filterwarnings('ignore')

### 1-1. 데이터 로드 및 합성 이상치(사고/정체) 주입

In [ ]:
df = pd.read_csv('c:\\AI_1team\\traffic_features.csv')
features = ['car_count', 'bus_count', 'truck_count', 'density']
X = df[features].values.astype(float)

# 임의 이상치 생성 (마지막 10%를 트래픽 폭주로 변환)
y_true = np.zeros(len(X))
anomaly_idx = int(len(X) * 0.9)
X[anomaly_idx:, 0] *= 3.5  
X[anomaly_idx:, 3] *= 4.0  
y_true[anomaly_idx:] = 1

train_size = int(len(X) * 0.7)
X_train, X_test = X[:train_size], X[train_size:]
y_test = y_true[train_size:]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### 1-2. 세 가지 모델 학습 및 에측 수행

In [ ]:
# 1. Autoencoder
def build_ae(latent_dim):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(4,)),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(latent_dim, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(4, activation=None)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

ae = build_ae(16)
ae.fit(X_train_s, X_train_s, epochs=30, batch_size=32, validation_split=0.1, verbose=0)
mse = np.mean(np.square(X_test_s - ae.predict(X_test_s, verbose=0)), axis=1)
threshold = np.percentile(mse, 80)
y_pred_ae = (mse > threshold).astype(int)

# 2. Isolation Forest
iso = IsolationForest(contamination=0.2, random_state=42)
preds_iso = iso.fit_predict(X_train_s)
preds_iso = iso.predict(X_test_s)
y_pred_iso = np.where(preds_iso == -1, 1, 0)

# 3. One-Class SVM
ocsvm = OneClassSVM(nu=0.2, gamma='scale')
ocsvm.fit(X_train_s)
preds_ocsvm = ocsvm.predict(X_test_s)
y_pred_ocsvm = np.where(preds_ocsvm == -1, 1, 0)

### 1-3. 벤치마크 결과 시각화 (Recall, F1-Score, ROC-AUC)

In [ ]:
models = ['Autoencoder', 'Isolation Forest', 'One-Class SVM']
y_preds = [y_pred_ae, y_pred_iso, y_pred_ocsvm]

results = []
for name, y_pred in zip(models, y_preds):
    results.append({
        'Model': name,
        'Recall (재현율)': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred)
    })

bench_df = pd.DataFrame(results)
display(bench_df)

# 바 차트 시각화
bench_df.set_index('Model').plot(kind='bar', figsize=(8,5), colormap='Set2')
plt.title('Anomaly Detection Benchmark (Recall & F1-Score)')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

# ROC Curve
iso_scores = -iso.decision_function(X_test_s)
ocsvm_scores = -ocsvm.decision_function(X_test_s)

plt.figure(figsize=(8,6))
for name, score in zip(models, [mse, iso_scores, ocsvm_scores]):
    fpr, tpr, _ = roc_curve(y_test, score)
    auc = roc_auc_score(y_test, score)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
    
plt.plot([0,1], [0,1], 'k--')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

### 1-4. Autoencoder 내부 잠재 공간(Latent Space) 분석 (PCA)
오토인코더가 데이터를 압축한 '은닉층'에서 정상 데이터와 돌발 데이터가 어떻게 구별되는지 2차원으로 축소하여 시각화합니다.

In [ ]:
from sklearn.decomposition import PCA

# 오토인코더의 인코더 부분(잠재 공간까지)만 추출하는 모델 생성
# ae.layers[1] 은 Latent 크기가 16인 Dense 레이어입니다.
encoder = tf.keras.Sequential(ae.layers[:2])

# 테스트 데이터에 대해 잠재 공간 벡터(Latent Vectors) 추출
latent_vectors = encoder.predict(X_test_s, verbose=0)

# 16차원의 벡터를 시각화를 위해 PCA를 사용해 2차원으로 축소
pca = PCA(n_components=2)
latent_pca = pca.fit_transform(latent_vectors)

# 데이터프레임으로 정리
pca_df = pd.DataFrame(latent_pca, columns=['PCA1', 'PCA2'])
pca_df['Label'] = np.where(y_test == 1, 'Anomaly', 'Normal')

# Scatter Plot 시각화
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Label', palette={'Normal': 'blue', 'Anomaly': 'red'}, 
                data=pca_df, alpha=0.6, edgecolor=None)

plt.title('Latent Space Visualization of Autoencoder (PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
print("해석: 빨간 점(Anomaly)과 파란 점(Normal)이 뚜렷하게 분리될수록 오토인코더가 이상 상황을 잘 구별해냈다는 뜻입니다.")